## Experience Engine (Jupyter)

Notebook counterpart to `main.py`: same agent (local **Qwen 0.6B**), tools, and session behavior. **Verbose agent traces** and replies scroll in the left pane; the **environment view** (game or car) updates on the right.

**Environment:** This notebook sets `EXPERIENCE_ENGINE_ACTIVE_ENV=game_environment` by default (sim game, no robot). Change it or edit `select_environment.config` to use the car.

**Run order:** (1) UI setup cell → (2) load agent → (3) attach handlers. Restart the kernel if you change environment selection after importing `agent`/`tools`.

Commands: `exit` / `quit` / `q` (closing sequence), `EXIT` (abort, no closing), `clear` (reset chat history).

Display routing lives in `gui_viewer.py`: **`terminal`** = OpenCV window (CLI), **`jupyter`** = widget callback. This notebook calls `set_display_mode("jupyter")` before the environment starts. You can also set `DISPLAY_MODE=jupyter` in `select_environment.config`.

In [1]:
import os
from pathlib import Path

# Default to sim game for Jupyter (avoids hanging on robot init). Must run before `import agent` / `import tools`.
os.environ.setdefault("EXPERIENCE_ENGINE_ACTIVE_ENV", "game_environment")

from IPython.display import display, HTML
import ipywidgets as widgets
import cv2
import numpy as np
from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "agent.py").exists():
    display(
        HTML(
            "<p><b>Error:</b> Jupyter working directory must be the experience-engine repo "
            "(the folder that contains <code>agent.py</code>).</p>"
        )
    )
    raise RuntimeError("Wrong working directory")

load_dotenv(ROOT / ".env")

from gui_viewer import set_display_mode, set_jupyter_image_handler

set_display_mode("jupyter")

game_img = widgets.Image(
    format="png",
    layout=widgets.Layout(
        width="420px",
        max_width="48%",
        border="1px solid #ccc",
        object_fit="contain",
    ),
)


def _push_env_frame(bgr: np.ndarray) -> None:
    ok, buf = cv2.imencode(".png", bgr)
    if ok:
        game_img.value = buf.tobytes()


set_jupyter_image_handler(_push_env_frame)

log_out = widgets.Output(
    layout=widgets.Layout(
        height="640px",
        overflow_y="scroll",
        flex="1",
        min_width="340px",
        border="1px solid #ddd",
        padding="8px",
    )
)

header = widgets.HTML(
    "<h3>Experience Engine</h3>"
    "<p>Left: model + tool traces. Right: live environment frame. Use <b>Send</b> or Enter.</p>"
)

user_in = widgets.Text(placeholder="You: ", layout=widgets.Layout(width="100%"))
send_btn = widgets.Button(description="Send", button_style="primary")
reset_game_btn = widgets.Button(description="Reset Game", button_style="warning")
reset_bare_btn = widgets.Button(description="Reset Bare Game", button_style="warning")

input_row = widgets.HBox(
    [user_in, send_btn, reset_game_btn, reset_bare_btn],
    layout=widgets.Layout(width="100%", gap="6px"),
)
row = widgets.HBox(
    [log_out, game_img],
    layout=widgets.Layout(width="100%", align_items="flex-start", gap="14px"),
)
ui = widgets.VBox([header, row, input_row])
display(ui)

In [2]:
from agent import create_conversational_agent
import main as main_mod
from tools.session_tools import get_session_control_signal, reset_session_control_signal
from tools import close_env

inject_image = main_mod.inject_image
session_id = "default_session"

with log_out:
    print("Initializing agent (Qwen + vector stores + environment)...")

agent, message_history = create_conversational_agent()

with log_out:
    print("Ready. Type a message and click Send (or press Enter).\n")

[Environment Loader] Selected environment: game_environment (EXPERIENCE_ENGINE_ACTIVE_ENV)
pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
[Agent] Using Nous/Qwen3 native backend (AGENT_TYPE=nous)
Initializing vector stores...
Initializing vector stores...
Loaded existing semantic vector store from .faiss_semantic
Loaded existing procedural vector store from .faiss_procedural
Loaded existing episodic vector store from .faiss_episodic
No documents found in working. Creating empty vector store.
Vector stores initialized.
Initializing game environment...
Environment view: Jupyter mode (OpenCV window disabled; use notebook image widget).
Initial game view captured and displayed.
Game environment initialized successfully.
[LLM] Loading local Qwen3-0.6B model...
[LLM] First load will download ~1.2GB model files...
[LLM] Attempting to use CUDA device...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'top_p', 'top_k', 'pad_token_id', 'repetition_penalty', 'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[LLM] CUDA device loaded successfully
[LLM] Qwen3-0.6B loaded successfully
[NousAgent] Loaded environment blurb: game_blurb.txt
[NousAgent] Loaded context_prompt.md
[NousAgent] Loaded episodic memory: session_close_reflection_2025_12_05
[NousAgent] Ready (37 tools, thinking=enabled)


In [3]:
def start_fresh_session() -> None:
    global agent, message_history
    agent, message_history = create_conversational_agent()
    with log_out:
        print("\n[Session restarted]\n")


def on_send(_=None) -> None:
    global agent, message_history
    text = user_in.value.strip()
    user_in.value = ""
    if not text:
        return

    with log_out:
        print(f"You: {text}\n")

    if text == "EXIT":
        with log_out:
            print("\n[Aborting session immediately without closing sequence]")
        close_env()
        send_btn.disabled = True
        user_in.disabled = True
        return

    if text.lower() == "clear":
        message_history.clear()
        with log_out:
            print("\n[Conversation history cleared]\n")
        return

    if text.lower() in ("exit", "quit", "q"):
        with log_out:
            main_mod.run_closing_sequence(agent, session_id, "User ended the session")
            print("Goodbye!")
        send_btn.disabled = True
        user_in.disabled = True
        return

    with log_out:
        response = agent.invoke(
            {"input": inject_image(text)},
            config={"configurable": {"session_id": session_id}},
        )
        print(f"\nAssistant: {response['output']}\n")
        print("-" * 60 + "\n")

    signal = get_session_control_signal()
    if signal["action"] == "end":
        reset_session_control_signal()
        with log_out:
            main_mod.run_closing_sequence(agent, session_id, "Agent ended the session")
            print("Goodbye!")
        send_btn.disabled = True
        user_in.disabled = True
    elif signal["action"] == "restart":
        reset_session_control_signal()
        with log_out:
            main_mod.run_closing_sequence(agent, session_id, "Agent restarting the session")
        start_fresh_session()


def _reset_game(factory_name: str) -> None:
    """Swap the game instance without touching the agent or chat history."""
    from environments.game_environment import game_tools
    from environments.game_environment.init_game import (
        create_random_two_walls_game,
        random_bare_game,
    )

    factories = {
        "two_walls": create_random_two_walls_game,
        "bare": random_bare_game,
    }
    factory = factories[factory_name]
    new_game = factory()
    game_tools.initialize_game(game=new_game)
    with log_out:
        print(f"\n[Game reset: {factory_name}]\n")


def on_reset_game(_=None) -> None:
    _reset_game("two_walls")


def on_reset_bare(_=None) -> None:
    _reset_game("bare")


send_btn.on_click(on_send)
user_in.on_submit(lambda _: on_send())
reset_game_btn.on_click(on_reset_game)
reset_bare_btn.on_click(on_reset_bare)

/tmp/ipykernel_328136/926939586.py:91: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  user_in.on_submit(lambda _: on_send())
